# 30-Day Readmission Prediction: Logistic Regression
**Author:** Nafisa Tasnia  
**Project:** DS4400 — 30 Day Readmission Prediction in Diabetic Patients  
**Dataset:** Diabetes 130-US Hospitals 1999-2008

In [ ]:
# Standard data manipulation and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Everything we need from sklearn for building and evaluating the logistic regression model
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay
)
from sklearn.pipeline import Pipeline

# Suppress warnings so the output stays clean and readable
import warnings
warnings.filterwarnings('ignore')

# SMOTE helps us deal with class imbalance by synthetically generating minority class samples.
# If it's not installed we'll just fall back to class_weight='balanced' inside the model itself.
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    SMOTE_AVAILABLE = False
    print("imbalanced-learn not installed — will use class_weight='balanced' instead.")
    print("Install with: pip install imbalanced-learn")

print('All imports successful.')

In [ ]:
# Point this at wherever the CSV lives relative to where you run the notebook
DATA_PATH = 'diabetes+130-us+hospitals+for+years+1999-2008/diabetic_data.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')  
df.head(3)

## 3. Exploratory Data Analysis

In [ ]:
# Quick look at column types and null counts — useful before any cleaning
df.info()

In [ ]:
# The original target column has three values: 'NO', '>30', and '<30'.
# Since we only care about readmissions within 30 days, we'll convert this
# into a binary label: 1 if readmitted <30 days, 0 for everything else.
print('Target value counts:')
print(df['readmitted'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left plot: original 3-class distribution so we can see what we're starting with
df['readmitted'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon', 'green'])
axes[0].set_title('Original readmitted labels')
axes[0].set_xlabel('Readmitted')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Right plot: our binarized version — this is the actual prediction target
df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)
df['readmitted_30'].value_counts().plot(kind='bar', ax=axes[1], color=['steelblue', 'salmon'])
axes[1].set_title('Binary target: readmitted within 30 days')
axes[1].set_xlabel('0 = Not readmitted <30d, 1 = Readmitted <30d')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the imbalance ratio — this will inform how we handle it later
pos = df['readmitted_30'].sum()
total = len(df)
print(f'\nClass balance: {pos}/{total} = {pos/total*100:.1f}% positive (readmitted <30d)')

In [ ]:
# This dataset uses '?' as a placeholder for missing values instead of NaN,
# so we need to manually count those before deciding how to handle each column.
missing_counts = (df == '?').sum()
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing_counts, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print('Columns with ? missing values:')
print(missing_df)
# weight is nearly 97% missing — not worth keeping
# payer_code and medical_specialty are ~40-50% missing — also dropping those

In [ ]:
# Let's visualize the numerical features to understand their distributions before we scale.
# Things like num_procedures and number_inpatient tend to be heavily right-skewed,
# which is totally expected in healthcare data.
num_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
            'num_medications', 'number_outpatient', 'number_emergency',
            'number_inpatient', 'number_diagnoses']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, num_cols):
    df[col].hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('')
plt.suptitle('Numerical Feature Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
# 4.1  Drop columns we can't use

# weight: 97% missing, not recoverable — drop it
# payer_code: ~40% missing and insurance type isn't a clinical predictor we want to rely on
# medical_specialty: ~49% missing and would add noise more than signal at this level
# encounter_id / patient_nbr: just identifier columns, no predictive value
DROP_COLS = ['encounter_id', 'patient_nbr', 'weight', 'payer_code', 'medical_specialty']
df.drop(columns=DROP_COLS, inplace=True)

# We already created readmitted_30, so the original text column can go
df.drop(columns=['readmitted'], inplace=True)

print(f'Shape after dropping columns: {df.shape}')

In [ ]:
#  Handle missing values 

# Convert '?' to proper NaN so pandas knows they're missing
df.replace('?', np.nan, inplace=True)

# Race is only ~2% missing, so filling with the mode (most common race) is safe
df['race'].fillna(df['race'].mode()[0], inplace=True)

# For the three diagnosis columns, a tiny fraction are missing — labeling them
# 'Unknown' keeps the row and lets the ICD mapping function handle it gracefully
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col].fillna('Unknown', inplace=True)

print('Remaining nulls:', df.isnull().sum().sum())
# Note: A1Cresult and max_glu_serum still have many nulls — we'll handle those
# with ordinal encoding in section 4.7 (treating missing as 'not tested')

In [ ]:
# 4.3  Remove patients who died or went to hospice 

# These patients literally cannot be readmitted, so including them
# would artificially inflate our '0' class and mislead the model.
# discharge_disposition_id codes: 11=Expired, 13=Hospice/home, 14=Hospice/facility,
# 19-21=Expired variants (Medicaid only)
exclude_discharge = [11, 13, 14, 19, 20, 21]
df = df[~df['discharge_disposition_id'].isin(exclude_discharge)].copy()
print(f'Shape after removing expired/hospice: {df.shape}')

In [ ]:
#  4.4  Note on duplicate patient encounters 

# Many patients appear multiple times in this dataset (multiple hospital stays).
# Ideally we'd deduplicate to one row per patient to avoid data leakage between
# visits, but since we already dropped encounter_id we can't do that cleanly here.
# We proceed with all valid encounters — something to flag as a limitation.
df_ids = pd.read_csv(DATA_PATH, usecols=['encounter_id', 'patient_nbr'])
df_ids = df_ids[~df_ids.index.isin(df_ids.index)].copy()  # placeholder — no actual dedup

print('Proceeding with all valid encounters (no patient dedup applied).')
print(f'Current shape: {df.shape}')

In [ ]:
# Map ICD-9 diagnosis codes to broad disease categories

# The raw ICD-9 codes are too granular for a logistic regression — there are thousands
# of unique values and most appear rarely. Grouping them into ~8 clinical categories
# gives the model something meaningful to work with.
def map_icd9(code):
    """Map an ICD-9 code string to a broad disease category."""
    if pd.isna(code) or code == 'Unknown':
        return 'Other'
    code_str = str(code).upper()

    # E-codes = external causes of injury; V-codes = supplemental factors
    # Neither maps cleanly to a disease category, so treat separately
    if code_str.startswith('E') or code_str.startswith('V'):
        return 'External/Supplemental'

    try:
        c = float(code_str)
    except ValueError:
        return 'Other'

    # ICD-9 numeric range mappings to clinical categories
    if 390 <= c <= 459 or c == 785:
        return 'Circulatory'
    elif 460 <= c <= 519 or c == 786:
        return 'Respiratory'
    elif 520 <= c <= 579 or c == 787:
        return 'Digestive'
    elif 250 <= c < 251:
        return 'Diabetes'        # 250.xx are the core diabetes codes
    elif 800 <= c <= 999:
        return 'Injury'
    elif 710 <= c <= 739:
        return 'Musculoskeletal'
    elif 580 <= c <= 629 or c == 788:
        return 'Genitourinary'
    elif 140 <= c <= 239:
        return 'Neoplasms'
    else:
        return 'Other'

# Apply to all three diagnosis columns (primary, secondary, tertiary)
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(map_icd9)

print('Diagnosis category counts (diag_1):')
print(df['diag_1'].value_counts())

In [ ]:
# 4.6  Encode medication columns 

# Each of the 23 medication columns records whether the drug dosage was:
# No (not prescribed), Steady (unchanged), Up (increased), or Down (decreased).
# We encode these as 0/1/2/3 — preserving the ordinal direction of change.
MED_COLS = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

med_map = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': 3}
for col in MED_COLS:
    # Any values that didn't match the map (shouldn't happen, but just in case)
    # get filled with 0, meaning 'not prescribed'
    df[col] = df[col].map(med_map).fillna(0).astype(int)

print('Medication columns encoded.')

In [ ]:
# Encode the remaining categorical columns 

# Simple binary flags: 'Ch' means a medication change was made, 'Yes' = on diabetes meds
df['change'] = (df['change'] == 'Ch').astype(int)
df['diabetesMed'] = (df['diabetesMed'] == 'Yes').astype(int)
# Female=1, Male=0 — straightforward binary
df['gender'] = (df['gender'] == 'Female').astype(int)

# A1C and glucose levels are ordinal — higher values = worse glycemic control.
# Missing (None) most likely means the test wasn't done, so we encode it as 0
# rather than dropping those rows.
a1c_map = {'None': 0, 'Norm': 1, '>7': 2, '>8': 3}
glu_map = {'None': 0, 'Norm': 1, '>200': 2, '>300': 3}
df['A1Cresult'] = df['A1Cresult'].map(a1c_map).fillna(0).astype(int)
df['max_glu_serum'] = df['max_glu_serum'].map(glu_map).fillna(0).astype(int)

# Age is given as a bracket like '[50-60)' — we convert to the midpoint (55)
# so the model can treat it as a continuous numeric variable
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}
df['age'] = df['age'].map(age_map).fillna(df['age'].map(age_map).median())

# Race and the three diagnosis category columns are nominal (no natural order),
# so one-hot encoding is the right approach here.
# drop_first=True avoids the dummy variable trap.
CAT_OHE = ['race', 'diag_1', 'diag_2', 'diag_3']
df = pd.get_dummies(df, columns=CAT_OHE, drop_first=True)

print(f'Shape after encoding: {df.shape}')
df.head(2)

## 5. Feature Engineering

In [ ]:
# 5.1  Create new features from existing ones 

# Patients with high prior utilization (many previous visits) are often sicker overall,
# which makes this a useful composite signal for readmission risk.
df['total_prior_visits'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']

# If many medications had their dosage changed during this stay, the patient's
# regimen may be unstable — which could relate to higher readmission risk.
med_change_cols = [c for c in MED_COLS if c in df.columns]
df['n_meds_changed'] = df[med_change_cols].apply(lambda row: (row != 0).sum(), axis=1)

# Total number of medications actively being used (any dosage level > 0).
# Higher complexity can indicate more severe disease.
df['med_complexity'] = df[med_change_cols].apply(lambda row: (row > 0).sum(), axis=1)

# How many procedures happened per day in hospital — a rough proxy for care intensity.
# We replace 0 time_in_hospital with 1 to avoid division by zero (shouldn't happen,
# but better safe than sorry).
df['procedures_per_day'] = df['num_procedures'] / df['time_in_hospital'].replace(0, 1)

print('New features created: total_prior_visits, n_meds_changed, med_complexity, procedures_per_day')

## 6. Train / Test Split

In [ ]:
TARGET = 'readmitted_30'
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Some boolean columns from get_dummies may have snuck through — cast everything to numeric
X = X.select_dtypes(include=[np.number])

# 80/20 split is standard for a dataset this size.
# stratify=y is important here because our classes are imbalanced (~11% positive);
# without it, we might get a luckily easy or unluckily hard test set by chance.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train positive rate: {y_train.mean():.3f}')  
print(f'Test  positive rate: {y_test.mean():.3f}')  

## 7. Handle Class Imbalance

In [ ]:
# Only 11% of patients were actually readmitted within 30 days.
# If we ignore this imbalance, the model will just predict 'not readmitted'
# for almost everything and still get 89% accuracy — which is useless.
#
# SMOTE (Synthetic Minority Over-sampling Technique) creates new synthetic examples
# of the minority class by interpolating between existing ones, so the model
# sees a more balanced training set.
#
# IMPORTANT: We only apply SMOTE to the training set — never the test set.
# The test set must reflect the real-world distribution to get honest metrics.
if SMOTE_AVAILABLE:
    sm = SMOTE(random_state=42)
    X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)
    print(f'After SMOTE — Train shape: {X_train_bal.shape}')
    print(f'Class balance: {pd.Series(y_train_bal).value_counts().to_dict()}')
else:
    # If SMOTE isn't available, we'll tell the model to penalize mistakes on
    # the minority class more heavily using class_weight='balanced'
    X_train_bal, y_train_bal = X_train, y_train
    print('Using class_weight="balanced" in the model instead of SMOTE.')

## 8. Feature Scaling

In [ ]:
# Logistic regression uses gradient-based optimization, so features on very different
# scales (e.g. age in the 50s vs. num_procedures near 0-6) can slow convergence
# or cause certain features to dominate unfairly.
# StandardScaler centers each feature at mean=0 with std=1 to fix this.
#
# Critical: fit the scaler only on training data, then apply that same transformation
# to the test set. Fitting on the test set would leak information about the test distribution.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_bal)  # fit + transform on training data
X_test_sc  = scaler.transform(X_test)           # only transform on test data
print('Scaling applied.')

## 9. Logistic Regression — Baseline Model

In [ ]:
# If SMOTE balanced the training data, we don't need class_weight='balanced' too —
# using both would over-correct. If SMOTE wasn't available, set it here instead.
class_weight_setting = None if SMOTE_AVAILABLE else 'balanced'

# Baseline model with mostly default settings — we'll tune this in section 10.
# lbfgs is a good general-purpose solver for medium-sized datasets.
# max_iter=1000 gives it plenty of room to converge — 100 is often not enough.
lr_base = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    class_weight=class_weight_setting,
    random_state=42
)
lr_base.fit(X_train_sc, y_train_bal)
print('Baseline model trained.')

In [ ]:
def evaluate_model(model, X_test_scaled, y_test, model_name='Model'):
    """
    Runs predictions and prints all our key metrics, plus a confusion matrix
    and ROC curve. Returns the metrics as a dict so we can compare models later.
    """
    y_pred = model.predict(X_test_scaled)
    # predict_proba gives us the probability of class 1, needed for ROC-AUC
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    acc   = accuracy_score(y_test, y_pred)
    f1    = f1_score(y_test, y_pred)
    prec  = precision_score(y_test, y_pred)
    rec   = recall_score(y_test, y_pred)
    roc   = roc_auc_score(y_test, y_prob)
    cm    = confusion_matrix(y_test, y_pred)

    print(f'\n===== {model_name} =====')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')  # of patients we flagged, how many were actually readmitted?
    print(f'  Recall    : {rec:.4f}')   # of patients who were actually readmitted, how many did we catch?
    print(f'  F1 Score  : {f1:.4f}')   # harmonic mean of precision and recall
    print(f'  ROC-AUC   : {roc:.4f}')  # overall discriminative power across all thresholds
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['Not <30d', 'Readmit <30d']))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Confusion matrix — lets us see the actual vs predicted breakdown
    # Top-left = true negatives, bottom-right = true positives
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Pred: Not <30d', 'Pred: <30d'],
                yticklabels=['True: Not <30d', 'True: <30d'], ax=axes[0])
    axes[0].set_title(f'{model_name} — Confusion Matrix')

    # ROC curve — the closer it hugs the top-left corner, the better
    RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1],
                                      name=model_name, color='steelblue')
    axes[1].set_title(f'{model_name} — ROC Curve')

    plt.tight_layout()
    fname = model_name.lower().replace(' ', '_') + '_eval.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

    return {'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec, 'roc_auc': roc}

baseline_metrics = evaluate_model(lr_base, X_test_sc, y_test, 'Logistic Regression (Baseline)')

## 10. Hyperparameter Tuning (GridSearchCV)

In [ ]:
# C controls regularization strength — smaller C = more regularization = simpler model.
# L1 penalty (lasso) can zero out irrelevant features entirely (built-in feature selection).
# L2 penalty (ridge) shrinks all coefficients but keeps them all non-zero.
# We test both to see which works better for this dataset.
# Note: L1 only works with the 'liblinear' and 'saga' solvers, not 'lbfgs'.
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
}

# StratifiedKFold keeps the class ratio consistent across folds —
# especially important with an imbalanced dataset like ours
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# We optimize for F1 rather than accuracy because accuracy alone is misleading
# when classes are imbalanced (a model that always predicts 0 still gets 89% accuracy)
grid_search = GridSearchCV(
    LogisticRegression(max_iter=2000, class_weight=class_weight_setting, random_state=42),
    param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,   # use all available CPU cores to speed this up
    verbose=1
)

grid_search.fit(X_train_sc, y_train_bal)

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV F1 score: {grid_search.best_score_:.4f}')

In [ ]:
# Grab the best model from the grid search and evaluate it on the held-out test set
best_lr = grid_search.best_estimator_
tuned_metrics = evaluate_model(best_lr, X_test_sc, y_test, 'Logistic Regression (Tuned)')

## 11. Cross-Validation on Tuned Model

In [ ]:
# Run 5-fold CV one more time on the final model to get a stable estimate of performance.
# The standard deviation across folds tells us how consistent the model is —
# high variance between folds could mean the model is sensitive to which data it sees.
cv_scores = cross_val_score(best_lr, X_train_sc, y_train_bal, cv=cv, scoring='f1', n_jobs=-1)

print(f'5-Fold CV F1 scores: {cv_scores}')
print(f'Mean CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

plt.figure(figsize=(6, 4))
plt.bar(range(1, 6), cv_scores, color='steelblue', edgecolor='white')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean = {cv_scores.mean():.3f}')
plt.xlabel('Fold')
plt.ylabel('F1 Score')
plt.title('5-Fold Cross-Validation F1 Scores')
plt.legend()
plt.tight_layout()
plt.savefig('cv_f1_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Feature Importance (Coefficients)

In [ ]:
# One of the big advantages of logistic regression over tree-based models is interpretability.
# Each coefficient tells us the change in log-odds of readmission for a 1-unit increase
# in that feature (after scaling, so units are standard deviations).
#
# Positive coefficient → feature increases readmission risk
# Negative coefficient → feature decreases readmission risk
feature_names = X.columns.tolist()
coefs = best_lr.coef_[0]

coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=False)  # rank by importance

top_n = 20
top_features = coef_df.head(top_n)

plt.figure(figsize=(9, 7))
# Color bars by direction: salmon = increases risk, blue = decreases risk
colors = ['salmon' if c > 0 else 'steelblue' for c in top_features['coefficient']]
plt.barh(top_features['feature'][::-1], top_features['coefficient'][::-1], color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.8)  # zero line for reference
plt.xlabel('Log-Odds Coefficient')
plt.title(f'Top {top_n} Feature Coefficients (Logistic Regression)')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 20 features by absolute coefficient:')
print(coef_df[['feature', 'coefficient']].head(20).to_string(index=False))

## 13. Summary Table

In [ ]:
# Side-by-side comparison to see how much tuning actually helped
summary = pd.DataFrame([
    {'Model': 'LR Baseline', **baseline_metrics},
    {'Model': 'LR Tuned',    **tuned_metrics},
]).set_index('Model').round(4)

print('\n===== Model Comparison =====')
print(summary.to_string())

summary.plot(kind='bar', figsize=(10, 5), colormap='Set2', edgecolor='white')
plt.title('Baseline vs Tuned Logistic Regression')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Key Findings

- **Target variable** was binarised as `1` if `readmitted == '<30'`, else `0`.
- The dataset is **heavily imbalanced** (~11% positive class); handled via SMOTE (or `class_weight='balanced'`).
- **Preprocessing steps:** dropped high-missingness columns (`weight`, `payer_code`, `medical_specialty`); imputed remaining nulls; encoded ICD-9 codes into broad categories; label-encoded medications and binary features; one-hot encoded race & diagnosis categories.
- **Feature engineering:** added `total_prior_visits`, `n_meds_changed`, `med_complexity`, and `procedures_per_day`.
- **Tuning** via 5-fold stratified GridSearchCV over `C ∈ {0.01, 0.1, 1, 10}` and L1/L2 penalty.
- **Strongest predictors** (see coefficient plot): prior inpatient admissions, number of diagnoses, time in hospital, and certain discharge/admission type codes.

> Logistic Regression provides a transparent, interpretable baseline well-suited for clinical settings where model explainability matters.